# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print metadata name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all record sets defined in the dataset using their `@id`. For each record set, we display its associated fields and their ids.

This provides a reference for extracting and analyzing data in later steps.

In [ ]:
# List all available record sets in the dataset
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, we'll extract all record sets into pandas DataFrames, referencing each by its `@id`.

In [ ]:
# Extract data from each record set into a dictionary of DataFrames
dataframes = {}
for record_set in dataset.record_sets:
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df
    print(f"Loaded {len(df)} records from record set {record_set.name} (@id: {record_set.id})")

# Choose the main survey record set for demonstration (replace the @id below as fits your use case)
main_record_set_id = None
for record_set in dataset.record_sets:
    if 'survey' in record_set.name.lower() or 'responses' in record_set.name.lower() or 'main' in record_set.name.lower():
        main_record_set_id = record_set.id
        break
# If not found by name heuristic, just pick the first record set
if not main_record_set_id:
    main_record_set_id = dataset.record_sets[0].id

print(f"\nColumns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We use the `@id` of fields to reference them precisely.

In [ ]:
# Choose a representative numeric field for filtering and normalization
# We'll try to use the GAD-7 total score if present, otherwise pick the first numeric column
numeric_field_id = None
group_field_id = None
main_fields = None
for record_set in dataset.record_sets:
    if record_set.id == main_record_set_id:
        main_fields = record_set.fields
        break

if main_fields:
    # Attempt to find a field relating to GAD-7
    for field in main_fields:
        if ('gad' in field.name.lower() and 'score' in field.name.lower()) or ('phq' in field.name.lower() and 'score' in field.name.lower()) or ('psq' in field.name.lower() and 'score' in field.name.lower()):
            if field.data_type in ('Float', 'Number', 'Integer', 'schema:Float', 'schema:Number', 'schema:Integer'):
                numeric_field_id = field.id
                break
    # If not found, simply pick the first numeric field
    if numeric_field_id is None:
        for field in main_fields:
            if field.data_type in ('Float', 'Number', 'Integer', 'schema:Float', 'schema:Number', 'schema:Integer'):
                numeric_field_id = field.id
                break
    # Try to pick a group-by field such as gender or education
    for field in main_fields:
        if group_field_id is None and any(key in field.name.lower() for key in ['gender', 'sex', 'education', 'village', 'marital']):
            group_field_id = field.id

print(f"Using numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Using group field: {group_field_id}")

# Filter records based on a threshold value for the chosen numeric field
df = dataframes[main_record_set_id]

if numeric_field_id in df.columns:
    # Replace threshold with an appropriate value for fair demonstration
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalizing the field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by group_field if available
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"No numeric field '{numeric_field_id}' found in the dataset to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we show a basic histogram and a boxplot of the chosen numeric field, grouped by the categorical field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No suitable numeric field found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides valuable insights into mental health indicators collected in Kilifi County, Kenya.
- Data loaded for each record set using `@id` references enables precise and maintainable analyses.
- Exploratory data analysis reveals possible distributions and group-wise statistics for core survey metrics (such as screening scores).
- Further analyses can be performed per research needs, leveraging the rich metadata and FAIR structure of this dataset.